In [24]:
import pandas as pd

In [25]:
summary = pd.read_csv('./runs_sweep/best_by_setting_backbone.csv')

In [26]:
sweep_results = pd.read_csv('./runs_sweep/sweep_results_running.csv')

In [27]:
sweep_results.columns

Index(['setting', 'backbone', 'run_dir', 'hidden_dim', 'depth', 'batch_size',
       'lr', 'dropout', 'best_epoch', 'best_val_rmse', 'test_rmse', 'test_mae',
       'test_nrmse', 'test_r2', 'all_rmse', 'all_mae', 'all_n', 'daytime_rmse',
       'daytime_mae', 'daytime_n', 'clear_like_rmse', 'clear_like_mae',
       'clear_like_n', 'cloudy_rmse', 'cloudy_mae', 'cloudy_n', 'peak_rmse',
       'peak_mae', 'peak_n', 'ramp_rmse', 'ramp_mae', 'ramp_n'],
      dtype='str')

In [28]:
summary

,setting,backbone,run_dir,hidden_dim,depth,batch_size,lr,dropout,best_epoch,best_val_rmse,...,clear_like_n,cloudy_rmse,cloudy_mae,cloudy_n,peak_rmse,peak_mae,peak_n,ramp_rmse,ramp_mae,ramp_n
0,decomposition,mlp,runs_sweep/decomposition_mlp__hd128__d3__bs32_...,128,3,32,0.0010,0.0,70,464.961936,...,801,687.924277,483.573375,91,725.984174,547.614183,184,652.778697,485.909636,184
1,direct,mlp,runs_sweep/direct_mlp__hd128__d4__bs32__lr0p00...,128,4,32,0.0010,0.2,57,458.920013,...,801,680.698146,524.740816,91,731.500314,586.190188,184,630.069016,467.130586,184
2,physics_feature,mlp,runs_sweep/physics_feature_mlp__hd128__d3__bs3...,128,3,32,0.0010,0.1,53,461.940650,...,801,601.727248,451.214844,91,770.863603,632.413164,184,618.477523,454.826803,184
3,decomposition,tcn,runs_sweep/decomposition_tcn__hd256__d3__bs64_...,256,3,64,0.0005,0.0,30,458.802164,...,801,701.506539,520.708383,91,756.520156,630.597356,184,633.999727,529.304062,184
4,direct,tcn,runs_sweep/direct_tcn__hd64__d3__bs32__lr0p000...,64,3,32,0.0005,0.2,44,450.624951,...,801,648.132075,479.250090,91,728.078731,568.350730,184,614.404216,466.984777,184
5,physics_feature,tcn,runs_sweep/physics_feature_tcn__hd128__d3__bs3...,128,3,32,0.0005,0.0,28,448.481942,...,801,562.958376,389.882176,91,718.648651,557.360551,184,646.715657,489.513163,184
6,decomposition,transformer,runs_sweep/decomposition_transformer__hd32__d1...,32,1,64,0.0002,0.1,76,553.372932,...,801,377.338126,257.007918,91,885.688797,602.758942,184,838.335806,644.005398,184
7,direct,transformer,runs_sweep/direct_transformer__hd32__d2__bs64_...,32,2,64,0.0005,0.3,200,492.956742,...,801,841.809494,649.253300,91,903.583974,807.522450,184,537.190148,377.678206,184
8,physics_feature,transformer,runs_sweep/physics_feature_transformer__hd64__...,64,2,64,0.0005,0.1,147,492.697680,...,801,504.421143,292.592782,91,901.503871,754.810230,184,640.658995,417.488947,184


In [29]:
summary.sort_values('test_rmse', ascending=True)[['setting', 'backbone', 'test_rmse']]

,setting,backbone,test_rmse
5,physics_feature,tcn,365.820719
0,decomposition,mlp,373.852808
1,direct,mlp,374.196274
3,decomposition,tcn,382.871936
2,physics_feature,mlp,386.036108
4,direct,tcn,388.904842
8,physics_feature,transformer,412.656598
7,direct,transformer,433.962982
6,decomposition,transformer,449.241287


In [30]:
sweep_results.sort_values('test_rmse', ascending=True)[['setting', 'backbone', 'test_rmse']].groupby(['setting', 'backbone']).mean()

test_rmse
setting         backbone                
decomposition   mlp           398.030226
                tcn           420.997612
                transformer   439.616892
direct          mlp           374.713305
                tcn           383.854339
                transformer  1052.413180
physics_feature mlp           372.150569
                tcn           378.773738
                transformer  1050.684602

In [31]:
from pathlib import Path

In [32]:
runs = [
    "runs/decomposition_mlp/predictions.csv",
    "runs/decomposition_tcn/predictions.csv",
    "runs/decomposition_transformer/predictions.csv",
]

for path in runs:
    path = Path(path)
    if not path.exists():
        print("[missing]", path)
        continue

    df = pd.read_csv(path)

    print("\n" + "="*80)
    print(path.parent.name)
    print(df[["y_true", "y_pred", "p_clear", "c_hat", "r_hat"]].describe())

    print("c_hat range:", df["c_hat"].min(), df["c_hat"].max())
    print("c_hat std:", df["c_hat"].std())
    print("r_hat abs mean:", df["r_hat"].abs().mean())
    print("r_hat abs max:", df["r_hat"].abs().max())
    print("r_hat std:", df["r_hat"].std())

    corr_c_p = df["c_hat"].corr(df["p_clear"])
    corr_c_y = df["c_hat"].corr(df["y_true"])
    corr_r_y = df["r_hat"].corr(df["y_true"])

    print("corr(c_hat, p_clear):", corr_c_p)
    print("corr(c_hat, y_true):", corr_c_y)
    print("corr(r_hat, y_true):", corr_r_y)


decomposition_mlp
            y_true       y_pred      p_clear        c_hat        r_hat
count  2305.000000  2305.000000  2305.000000  2305.000000  2305.000000
mean    889.142213   950.741521   466.720330     1.155634   422.428100
std    1306.362352  1314.967341   683.533334     0.170242   584.749896
min       0.000000     1.162963     0.000000     0.152265     1.156258
25%       0.000000    12.272667     0.000000     1.194897    11.323292
50%       0.000000    99.987602     0.000000     1.199884    49.094189
75%    1678.000000  1894.198853  1041.884644     1.200000   748.848328
max    4304.399902  4587.866699  2029.189575     1.200000  2458.437256
c_hat range: 0.1522654294967651 1.2000000476837158
c_hat std: 0.17024176836049087
r_hat abs mean: 422.4281002412908
r_hat abs max: 2458.437255859375
r_hat std: 584.7498958258618
corr(c_hat, p_clear): -0.06182192569672741
corr(c_hat, y_true): 0.11459996213972941
corr(r_hat, y_true): 0.8977221281848248

decomposition_tcn
            y_true   